# 1. Quickstart: your first SVD check

This notebook runs a complete SVD through `svdlab` and shows how to read the result.
It takes a few minutes on a laptop CPU. On a machine with a CUDA or ROCm GPU it also runs
the GPU part at the end.

**Before you start:** install the package once from the repository root
(`python -m pip install -e '.[laboratory]'`, see
[tutorial 1](../docs/tutorials/01-install-and-first-check.md)) and select that Python
environment as the notebook kernel. Then *Run → Run All Cells*.

In [1]:
# Setup: find the repository, choose the device. Run this cell first.
from pathlib import Path
import os, sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'svdlab' / '__init__.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))      # works even if the package is not installed with pip
os.chdir(ROOT)                         # all paths below are relative to the repository root

import torch
import svdlab

USE_GPU = True                         # set to False to stay on the CPU even when a GPU is present
DEVICE = 'cuda' if USE_GPU and torch.cuda.is_available() else 'cpu'
torch.set_num_threads(4)
print(f'torch {torch.__version__}, device: {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if DEVICE == 'cuda' else ''))

torch 2.14.1+cu130, device: cpu


## The contract

An SVD method is a function

```python
def svd(a, rank):          # a: torch tensor (m, n), real or complex; rank <= min(m, n)
    ...
    return U, s, Vh        # (m, k), (k,), (k, n) with k >= rank, s descending
```

The repository ships a working one to start from: `examples/my_svd.py`, a one-sided
Jacobi SVD in plain PyTorch. `svdlab.load_file` loads the function `svd` from a file,
exactly as the command line does.

In [2]:
svd = svdlab.load_file('examples/my_svd.py')

a = torch.randn(6, 4, dtype=torch.float64)
u, s, vh = svd(a, 4)
print('shapes:', tuple(u.shape), tuple(s.shape), tuple(vh.shape))
print('singular values:', s)
print('|A - U diag(s) Vh| =', float(torch.linalg.norm(a - u * s @ vh)))

shapes: (6, 4) (4,) (4, 4)
singular values: tensor([3.7178, 2.3377, 1.2391, 0.3527], dtype=torch.float64)
|A - U diag(s) Vh| = 2.5592762937726262e-15


## Run the check

`svdlab.check` runs the method on a fixed set of test matrices, compares every result with
an FP64 reference SVD and times the call. `QUICK_SUITE` is six small matrices for a first
run; `DEFAULT_SUITE` (the default) has eight larger ones.

In [3]:
rows = svdlab.check(svd, svdlab.QUICK_SUITE, name='my_svd', device=DEVICE)

my_svd               gaussian_tall          ok      59.275 ms  recon 4.51e-15  tail× -  σ-rel 3.64e-15  orth 4.66e-15


my_svd               gaussian_wide          ok      43.905 ms  recon 4.34e-15  tail× -  σ-rel 3.71e-15  orth 4.25e-15


my_svd               ill_conditioned_1e8    ok      80.384 ms  recon 6.48e-15  tail× -  σ-rel 4.24e-10  orth 6.60e-15


my_svd               clustered              ok      75.449 ms  recon 8.67e-15  tail× -  σ-rel 4.75e-14  orth 8.29e-15


my_svd               complex_gaussian       ok      65.306 ms  recon 3.04e-15  tail× -  σ-rel 2.44e-15  orth 3.20e-15


my_svd               low_rank_k4            ok     181.505 ms  recon 6.64e-06  tail× 1.00e+00  σ-rel 3.99e-15  orth 7.96e-15


### How to read a line

```text
my_svd   gaussian_tall   ok   64.858 ms   recon 4.51e-15   tail× -   σ-rel 3.64e-15   orth 4.66e-15
```

| Field | Meaning |
| --- | --- |
| `gaussian_tall` | the test matrix (48×32 Gaussian, float64) |
| `ok` / `FAIL` | the verdict: errors within 100·eps·max(m, n) for this dtype and size |
| `ms` | median time of one call; only your function is timed |
| `recon` | ‖A − U diag(s) Vᴴ‖ / ‖A‖ |
| `tail×` | truncated cases only: residual / best possible rank-k residual (1.00 is optimal) |
| `σ-rel` | largest relative error of a singular value |
| `orth` | how far U and V are from orthonormal |

`ill_conditioned_1e8` has a much larger `σ-rel` and is still `ok`: its smallest singular
values are 1e8 times smaller than the largest, and no float64 method gets them to full
relative accuracy. `low_rank_k4` asks for 4 triplets only, so `recon` is the size of the
part left out, and `tail× 1.00` says that is optimal.
[Tutorial 5](../docs/tutorials/05-metrics-and-matrices.md) explains every metric.

`check` returns one dictionary per matrix. `svdlab.table` turns them into a DataFrame
with every metric:

In [4]:
svdlab.table(rows)[['case', 'm', 'n', 'dtype', 'rank', 'ok', 'time_median_ms',
                    'reconstruction_relative_fro', 'resolved_spectrum_max_relative_error',
                    'orthogonality_u_fro_over_sqrt_cols']]

,case,m,n,dtype,rank,ok,time_median_ms,reconstruction_relative_fro,resolved_spectrum_max_relative_error,orthogonality_u_fro_over_sqrt_cols
0,gaussian_tall,48,32,float64,32,True,59.275111,4.507216e-15,3.641159e-15,1.545500e-15
1,gaussian_wide,24,40,float64,24,True,43.904999,4.339813e-15,3.708041e-15,4.246111e-15
2,ill_conditioned_1e8,40,30,float64,30,True,80.383865,6.480045e-15,4.239494e-10,2.221803e-15
3,clustered,40,30,float64,30,True,75.449324,8.666571e-15,4.748805e-14,1.903608e-15
4,complex_gaussian,32,20,complex128,20,True,65.305873,3.042918e-15,2.440787e-15,1.342602e-15
5,low_rank_k4,80,60,float64,4,True,181.505022,6.635512e-06,3.993729e-15,4.910392e-16


## Compare with other methods

`compare` runs several methods on the same matrices. A method can be a function, a file
name or a builtin name: `'torch'` is `torch.linalg.svd`, `'jacobi'` the repository's own
Jacobi SVD (`python -m svdlab list` prints all of them).

In [5]:
rows = svdlab.compare({'my_svd': svd, 'torch': 'torch', 'jacobi': 'jacobi'}, svdlab.QUICK_SUITE,
                      device=DEVICE, verbose=False)
for s in svdlab.summary(rows):
    print(f"{s['method']:<8} {s['ok']}/{s['cases']} ok   worst recon {s['worst_reconstruction']:.1e}   "
          f"total {s['total_median_ms']:8.2f} ms")

my_svd   6/6 ok   worst recon 6.6e-06   total   554.30 ms
torch    6/6 ok   worst recon 6.6e-06   total     1.10 ms
jacobi   6/6 ok   worst recon 6.6e-06   total   507.03 ms


In [6]:
svdlab.table(rows).pivot(index='case', columns='method', values='time_median_ms').round(3)

method,jacobi,my_svd,torch
case,,,
clustered,84.478,65.452,0.120
complex_gaussian,55.949,35.469,0.141
gaussian_tall,53.206,80.553,0.303
gaussian_wide,39.184,36.560,0.127
ill_conditioned_1e8,100.038,101.012,0.139
low_rank_k4,174.173,235.254,0.271


## The same from the terminal

Everything above is one command in a terminal, run from the repository root:

```bash
python -m svdlab check examples/my_svd.py --quick --compare torch,jacobi
```

The cell below runs exactly that command inside the notebook. The exit code is 0 when
every case of `my_svd` is `ok`, 1 when one fails (methods after `--compare` never change
it), so the command can be used in CI.

In [7]:
import shlex
from svdlab.__main__ import main as svdlab_main

def svdlab_cli(command):
    """Run `python -m svdlab <command>` inside this notebook (same output, same exit code)."""
    code = svdlab_main(shlex.split(command))
    print(f'exit code: {code}')

In [8]:
svdlab_cli('check examples/my_svd.py --quick --compare torch,jacobi')

recon  = |A - U diag(s) Vh| / |A|          tail× = recon / best possible (truncated cases)
σ-rel  = max relative error of the values   orth  = max(|UᴴU - I|, |V Vᴴ - I|) / sqrt(k)
ok     = orth and recon within 100·eps·max(m, n), or tail× <= 1.05



my_svd               gaussian_tall          ok      53.231 ms  recon 4.51e-15  tail× -  σ-rel 3.64e-15  orth 4.66e-15


my_svd               gaussian_wide          ok      39.845 ms  recon 4.34e-15  tail× -  σ-rel 3.71e-15  orth 4.25e-15


my_svd               ill_conditioned_1e8    ok      88.622 ms  recon 6.48e-15  tail× -  σ-rel 4.24e-10  orth 6.60e-15


my_svd               clustered              ok      69.248 ms  recon 8.67e-15  tail× -  σ-rel 4.75e-14  orth 8.29e-15


...

jacobi               gaussian_wide          ok      68.119 ms  recon 5.65e-15  tail× -  σ-rel 4.71e-15  orth 5.24e-15


jacobi               ill_conditioned_1e8    ok     103.341 ms  recon 7.45e-15  tail× -  σ-rel 1.51e-09  orth 8.77e-15


jacobi               clustered              ok     131.181 ms  recon 1.02e-14  tail× -  σ-rel 1.99e-14  orth 9.73e-15


jacobi               complex_gaussian       ok      36.199 ms  recon 3.55e-15  tail× -  σ-rel 2.95e-15  orth 3.63e-15


jacobi               low_rank_k4            ok     184.333 ms  recon 6.64e-06  tail× 1.00e+00  σ-rel 4.64e-15  orth 9.03e-15

my_svd: 6/6 ok, total median 485.665 ms
torch: 6/6 ok, total median 13.288 ms
jacobi: 6/6 ok, total median 576.645 ms
exit code: 0


## On the GPU (optional)

`gpu_check` runs the method on the CPU and on the GPU. A case passes when both runs are
`ok`, the result really comes back on the GPU and the singular values agree. The cell
does nothing on a CPU-only machine. Terminal: `python -m svdlab check examples/my_svd.py --quick --gpu-check`.

In [9]:
if DEVICE == 'cuda':
    gpu_rows = svdlab.gpu_check(svd, svdlab.QUICK_SUITE, verbose=False)
    display(svdlab.table(gpu_rows)[['case', 'passed', 'cpu_ok', 'gpu_ok', 'output_device', 'speedup']])
else:
    print('No CUDA/ROCm device visible to torch: GPU check skipped.')

No CUDA/ROCm device visible to torch: GPU check skipped.


## Next

- [02_write_your_own_svd.ipynb](02_write_your_own_svd.ipynb): write an SVD, see it fail, fix it.
- [03_blocks_and_recipes.ipynb](03_blocks_and_recipes.ipynb): replace one step of an existing SVD.
- [04_native_cpp_cuda.ipynb](04_native_cpp_cuda.ipynb): C++ and CUDA.
- [05_metrics_and_matrices.ipynb](05_metrics_and_matrices.ipynb): every metric, your own matrices.